In [ ]:
pip install transformers torch accelerate einops

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Set the model to evaluation mode
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

In [ ]:
import pandas as pd
import re
from tqdm import tqdm

df = pd.read_csv('/content/drive/Shareddrives/ROARs/Data/labeled_dataset.csv')

def scoring(text):
    """
    helper to find the scores the model outputs - looks for the numbers following the labels.
    """
    # find all digits in the response
    digits = re.findall(r'\d', text)
    # expects 4 digits
    if len(digits) >= 4:
        return [int(d) for d in digits[:4]]
    else:
        # capture errored outputs
        return [None, None, None, None]

def grade_all_components(row):
    prompt = f""" Analyze these four sections of an academic report using a WEIGHTED AUDIT approach.

CRITICALITY & WEIGHTING: 1. Methods (30% Weight): Highest priority. Scrutinize strictly for direct measures and rubrics.
2. Results (30% Weight): Highest priority. Scrutinize strictly for numeric data and alignment.
3. PLO (25% Weight): Important priority. Must be a measurable learner-centric skill.
4. Improvement Plan (15% Weight): Supporting priority. Must be an actionable change.

Analyze these four sections of an academic report.
For each section, determine if it passes (1) or fails (0).
A 'Pass' means the section is complete, specific, and matches the context.

Acceptable possible criteria for the plo section:
1) Focus on a result to be achieved rather than an action (program, process, etc.) to be implemented.
2) Write PLOs/outcome statements using action verbs (see Bloom’s Taxonomy discussion below).
3) Programs should write PLOs from the learner perspective (administrative units should focus on specific, measurable operational goals/objectives in writing outcome statements).
4) Write PLOs/outcome statements in specific, measurable terms.
5) Strive to focus on a single achievement/accomplishment in the PLO/outcome statement rather than multiple (to more clearly align measures)

Acceptable possible criteria for the methods section:
1) Entry should explain why the assessment method changed from the
previous plan (e.g., used a different course, used a different assignment,
etc.)
2) The method entry must be aligned with the PLO that is being assessed
and assess the corresponding student knowledge, skills, or abilities.
3) Entry should outline what data was collected, from what sources, using
what methods (direct assessment), by whom, in what approximate
timeframe, and what rubric/score/criteria was used to determine PLO
achievement.
4) There should be a clear statement of how they evaluate student work –
what kind of scoring is used, and the expected target/standard of
achievement for minimally acceptable performance
5) The assessment methodology must use a direct measure of student
learning via a student work product (a performance, a written paper,
exam questions, etc.).


Acceptable possible criteria for the results sections:
1) Entry should not use the term “course grade” or “course grades”
however, assignment grades are ok provided they align with the stated
evaluation scale
2) Entry should only contain results/data from the current Academic
Year—not prior year data.
3) Entry should state who was assessed, what direct measure (student
artifact) was used, and data that reflects students’ results for the direct
assessment measure.
4) Entry should include a statement of achievement or non-achievement
of the PLO target/standard for achievement set in their assessment
plan. Entry can include a statement expressing inconclusive data but
must also include reasoning for the inconclusive data.
5) Entry should include an analysis and interpretation of the data and what
the data means in terms of PLO achievement.

1. PLO: {row['plo']}
2. Methods: {row['methods']}
3. Results: {row['results_conclusions']}
4. Improvement Plan: {row['improvement_plan']}

Respond ONLY in this format:
PLO: [0 or 1]
Methods: [0 or 1]
Results: [0 or 1]
Plan: [0 or 1]
"""
    messages = [
        {"role": "system", "content": "You are a precise academic auditor. You only output 0 or 1 for specific categories."},
        {"role": "user", "content": prompt}
    ]


    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=50,
        temperature=0.01
    )

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    result_text = response.split("assistant\n")[-1].strip()

    return scoring(result_text)

#print("Testing Row 1...")
#test_result = grade_all_components(df.iloc[0])
#print(f"Test Success! Row 1 Scores: {test_result}")


# run process
print("Generating individual component scores...")
tqdm.pandas()

# apply function then expand result into 4 new scoring features
scores = df.progress_apply(grade_all_components, axis=1)
df[['plo_score', 'methods_score', 'results_score', 'plan_score']] = pd.DataFrame(scores.tolist(), index=df.index)

df.to_csv('scored_results.csv', index=False)

# Show a preview of the new columns
print("\n--- Audit Sample ---")
print(df[['plo_score', 'methods_score', 'results_score', 'plan_score']].head())

Generating individual component scores...


  3%|▎         | 1/39 [00:00<00:00, 1433.46it/s]


NameError: name 'tokenizer' is not defined

In [ ]:
import torch
print(f"Is GPU available? {torch.cuda.is_available()}")